# nCoV PCA, t-SNE, and EOT analysis

In [ ]:
from pathlib import Path

import numpy as np

from ne_aligned_utils.analysis import (
    load_ncov_analysis_data,
    plot_embedding_panels,
    plot_joint_embedding,
    plot_joint_embedding_pair,
    plot_matrix,
    plot_ot_tsne_sweep_grid,
)
from ne_aligned_utils.methods import (
    build_ot_tsne_four_block_matrix,
    compute_eot_embedding_from_arrays,
    compute_eot_embedding_from_plan,
    compute_ot_plan,
    compute_pca_embedding_from_arrays,
    compute_seurat_embedding_from_arrays,
    compute_tsne_embedding,
    normalize_ot_tsne_graph,
    run_tsne_from_graph,
    sparsify_four_block_top_percentage,
)


In [ ]:
ncov = load_ncov_analysis_data(Path("data"))
ncov.dataset_names, ncov.short_dataset_names


In [ ]:
pca_embeddings = compute_pca_embedding_from_arrays(ncov.x1, ncov.x2, embd_dim=50, random_state=0)
fig, _ = plot_embedding_panels(pca_embeddings, ncov, color_by="batch")


In [ ]:
fig, _ = plot_embedding_panels(pca_embeddings, ncov, color_by="cell_type")


In [ ]:
tsne_embeddings = compute_tsne_embedding(pca_embeddings, random_state=0)
fig, _ = plot_embedding_panels(tsne_embeddings, ncov, color_by="batch")


In [ ]:
fig, _ = plot_embedding_panels(tsne_embeddings, ncov, color_by="cell_type")


In [ ]:
paper_eot_eps_times = 1.0  

eot_embeddings = compute_eot_embedding_from_arrays(
    ncov.x1,
    ncov.x2,
    t=0,
    eps_times=paper_eot_eps_times,
)
fig, _ = plot_embedding_panels(
    eot_embeddings,
    ncov,
    color_by="batch",
    point_size=1.5,
    alpha=0.45,
    standardize=True,
    aspect="auto",
)


In [ ]:
fig, _ = plot_embedding_panels(
    eot_embeddings,
    ncov,
    color_by="cell_type",
    point_size=1.5,
    alpha=0.45,
    standardize=True,
    aspect="auto",
)


### OT-TSNE

#### Step 1: Calculate OT Plan

In [ ]:
ot_plan = compute_ot_plan(ncov.x1, ncov.x2, eps_times=1.0, tol=1e-12, max_iter=int(1e4))
print(f"shape={ot_plan.shape}, sum={ot_plan.sum():.6f}, nnz={np.count_nonzero(ot_plan)}")
fig, _ = plot_matrix(ot_plan, title="OT plan")


#### Step 2: Build 4 Block Matrix

In [ ]:
ot_tsne_four_block = build_ot_tsne_four_block_matrix(ot_plan)
print(
    f"shape={ot_tsne_four_block.shape}, sum={ot_tsne_four_block.sum():.6f}, "
    f"nnz={np.count_nonzero(ot_tsne_four_block)}"
)
fig, _ = plot_matrix(ot_tsne_four_block, title="OT-tSNE 4-block matrix")


#### Step 3: Sparsify 4 Block Matrix (top 15% neighbors)

In [ ]:
ot_tsne_sparse_four_block = sparsify_four_block_top_percentage(
    ot_tsne_four_block,
    n0=ncov.x1.shape[0],
    n1=ncov.x2.shape[0],
    percentage=0.005,
    ensure_nonempty=True,
    binarize=True,
)
print(
    f"shape={ot_tsne_sparse_four_block.shape}, sum={ot_tsne_sparse_four_block.sum():.6f}, "
    f"nnz={np.count_nonzero(ot_tsne_sparse_four_block)}"
)
fig, _ = plot_matrix(ot_tsne_sparse_four_block, title="Sparse OT-tSNE 4-block matrix (binarized)")


#### Step 4: Normalize

In [ ]:
ot_tsne_graph = normalize_ot_tsne_graph(ot_tsne_sparse_four_block)
print(f"shape={ot_tsne_graph.shape}, sum={ot_tsne_graph.sum():.6f}, nnz={np.count_nonzero(ot_tsne_graph)}")
fig, _ = plot_matrix(ot_tsne_graph, title="Normalized OT-tSNE graph")


#### Step 5: t-SNE

In [ ]:
ot_tsne_embedding = run_tsne_from_graph(ot_tsne_graph, random_state=0, n_epochs=200)
ot_tsne_embedding.shape


In [ ]:
fig, _ = plot_joint_embedding_pair(
    ot_tsne_embedding,
    ncov,
    point_size=2.0,
    alpha=0.7,
    aspect="auto",
)
fig


#### OT-tSNE Sweep

In [ ]:
ot_tsne_sweep_embeddings, fig, _ = plot_ot_tsne_sweep_grid(
    ncov,
    percentages=(0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.4),
    eps_values=(0.001, 0.1, 1.0, 10.0, 100.0, 300.0),
    point_size=1.5,
    alpha=0.7,
    binarize=True,
    aspect="auto",
)


In [ ]:
ot_tsne_sweep_embeddings, fig, _ = plot_ot_tsne_sweep_grid(
    ncov,
    percentages=(0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.4),
    eps_values=(0.001, 0.1, 1.0, 10.0, 100.0, 300.0),
    point_size=1.5,
    alpha=0.7,
    binarize=False,
    aspect="auto",
)

### Embedding Metrics


In [ ]:
from pathlib import Path

import numpy as np

from ne_aligned_utils.methods import (
    compute_eot_embedding_from_arrays,
    compute_ot_tsne_embedding_from_arrays,
)

embedding_metric_percentages = (0.001, 0.005, 0.01, 0.05, 0.1, 0.15, 0.2)
embedding_metric_eps_values = (1.0, 10.0, 100.0, 300.0)
embedding_metric_k = 50
embedding_metric_random_state = 0
embedding_metric_n_epochs = 200
embedding_metric_binarize = True
embedding_metric_eot_t = 0
embedding_metric_eot_source = "raw_arrays"

embedding_metric_cache_dir_candidates = (Path("data/nCoV"), Path("../data/nCoV"))
embedding_metric_cache_dir = next(
    (path for path in embedding_metric_cache_dir_candidates if path.exists()),
    embedding_metric_cache_dir_candidates[0],
)
embedding_metric_cache_dir.mkdir(parents=True, exist_ok=True)
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = "binarized" if embedding_metric_binarize else "weighted"
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)

expected_shape = (
    len(embedding_metric_eps_values),
    len(embedding_metric_percentages),
    ncov.joint_data.shape[0],
    2,
)
expected_eot_shape = (len(embedding_metric_eps_values), ncov.joint_data.shape[0], 2)

if embedding_metric_cache_path.exists():
    with np.load(embedding_metric_cache_path) as cache:
        cached_eps_values = tuple(float(value) for value in cache["eps_values"])
        cached_percentages = tuple(float(value) for value in cache["percentages"])
        cached_binarize = bool(cache["binarize"].item())
        cached_eot_t = int(cache["eot_t"])
        cached_eot_source = str(cache["eot_source"].item()) if "eot_source" in cache else ""
        ot_tsne_metric_embeddings = np.asarray(cache["ot_tsne_embeddings"], dtype=float)
        eot_metric_embeddings = np.asarray(cache["eot_embeddings"], dtype=float)

    cache_matches_request = (
        cached_eps_values == embedding_metric_eps_values
        and cached_percentages == embedding_metric_percentages
        and cached_binarize == embedding_metric_binarize
        and cached_eot_t == embedding_metric_eot_t
        and cached_eot_source == embedding_metric_eot_source
        and ot_tsne_metric_embeddings.shape == expected_shape
        and eot_metric_embeddings.shape == expected_eot_shape
    )
else:
    cache_matches_request = False

if not cache_matches_request:
    print("Cache missing or incompatible; computing joint OT-tSNE metric embeddings...")
    ot_tsne_metric_embeddings = np.empty(expected_shape, dtype=float)
    eot_metric_embeddings = np.empty(expected_eot_shape, dtype=float)
    total_progress_steps = len(embedding_metric_eps_values) * (
        1 + len(embedding_metric_percentages)
    )
    progress_step = 0

    for eps_idx, eps_times in enumerate(embedding_metric_eps_values):
        progress_step += 1
        print(
            f"[{progress_step}/{total_progress_steps}] Computing paper-style EOT t={embedding_metric_eot_t} "
            f"from raw arrays for eps_times={eps_times:g}"
        )
        eot_metric_embeddings[eps_idx] = compute_eot_embedding_from_arrays(
            ncov.x1,
            ncov.x2,
            t=embedding_metric_eot_t,
            eps_times=eps_times,
        )

        for pct_idx, percentage in enumerate(embedding_metric_percentages):
            progress_step += 1
            print(
                f"[{progress_step}/{total_progress_steps}] Computing joint OT-tSNE embedding "
                f"for eps_times={eps_times:g}, top {100.0 * percentage:g}% neighbors"
            )
            ot_tsne_metric_embeddings[eps_idx, pct_idx] = (
                compute_ot_tsne_embedding_from_arrays(
                    ncov.x1,
                    ncov.x2,
                    eps_times=eps_times,
                    percentage=percentage,
                    binarize=embedding_metric_binarize,
                    random_state=embedding_metric_random_state,
                    n_epochs=embedding_metric_n_epochs,
                )
            )

    np.savez_compressed(
        embedding_metric_cache_path,
        eps_values=np.asarray(embedding_metric_eps_values, dtype=float),
        percentages=np.asarray(embedding_metric_percentages, dtype=float),
        ot_tsne_embeddings=ot_tsne_metric_embeddings,
        eot_embeddings=eot_metric_embeddings,
        binarize=np.asarray(embedding_metric_binarize),
        metric_k=np.asarray(embedding_metric_k, dtype=int),
        eot_t=np.asarray(embedding_metric_eot_t, dtype=int),
        eot_source=np.asarray(embedding_metric_eot_source),
        random_state=np.asarray(embedding_metric_random_state, dtype=int),
        n_epochs=np.asarray(embedding_metric_n_epochs, dtype=int),
    )
    print(f"Saved joint OT-tSNE metric embeddings to {embedding_metric_cache_path}")
else:
    print(f"Loaded joint OT-tSNE metric embeddings from {embedding_metric_cache_path}")

print(
    f"variant={embedding_metric_variant}, eps_times={embedding_metric_eps_values}, "
    f"percentages={embedding_metric_percentages}, k={embedding_metric_k}, "
    f"EOT source={embedding_metric_eot_source}, "
    f"OT grid shape={ot_tsne_metric_embeddings.shape}, "
    f"EOT grid shape={eot_metric_embeddings.shape}"
)


In [ ]:
from pathlib import Path

import numpy as np

from ne_aligned_utils.methods import compute_eot_embedding_from_arrays

embedding_metric_cache_dir_candidates = (Path("data/nCoV"), Path("../data/nCoV"))
embedding_metric_cache_dir = next(
    (path for path in embedding_metric_cache_dir_candidates if path.exists()),
    embedding_metric_cache_dir_candidates[0],
)
embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the previous embedding cache cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)

embedding_metric_selected_eot_t = 0  # Change this before running the cell.
embedding_metric_selected_eot_source = "raw_arrays"

embedding_metric_selected_eot_embeddings = np.empty(
    (len(embedding_metric_eps_values), ncov.joint_data.shape[0], 2),
    dtype=float,
)
for eps_idx, eps_times in enumerate(embedding_metric_eps_values):
    print(
        f"[{eps_idx + 1}/{len(embedding_metric_eps_values)}] Computing paper-style EOT t="
        f"{embedding_metric_selected_eot_t} from raw arrays for eps_times={eps_times:g}"
    )
    embedding_metric_selected_eot_embeddings[eps_idx] = compute_eot_embedding_from_arrays(
        ncov.x1,
        ncov.x2,
        t=embedding_metric_selected_eot_t,
        eps_times=eps_times,
    )

embedding_metric_selected_variant = embedding_metric_variant
embedding_metric_selected_eps_values = embedding_metric_eps_values.copy()

print(
    f"Prepared paper-style EOT reference embeddings for t={embedding_metric_selected_eot_t} "
    f"with shape {embedding_metric_selected_eot_embeddings.shape}"
)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from ne_aligned_utils.methods import compute_pca_embedding_from_arrays, compute_tsne_embedding
from ne_aligned_utils.benchmarking import (
    compute_shared_dataset_scale_bounds,
    plot_metric_heatmap_grid,
)
from ne_aligned_utils.metrics import (
    compute_knn_other_batch_fraction,
    compute_knn_same_cell_type_fraction,
)

embedding_metric_cache_dir_candidates = (Path("data/nCoV"), Path("../data/nCoV"))
embedding_metric_cache_dir = next(
    (path for path in embedding_metric_cache_dir_candidates if path.exists()),
    embedding_metric_cache_dir_candidates[0],
)
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the previous cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)
    embedding_metric_percentages = np.asarray(cache["percentages"], dtype=float)
    ot_tsne_metric_embeddings = np.asarray(cache["ot_tsne_embeddings"], dtype=float)
    embedding_metric_binarize = bool(cache["binarize"].item())
    embedding_metric_k = int(cache["metric_k"])
    embedding_metric_random_state = int(cache["random_state"]) if "random_state" in cache else 0

expected_selected_eot_source = "raw_arrays"
eot_metric_embeddings = globals().get("embedding_metric_selected_eot_embeddings")
if eot_metric_embeddings is None:
    raise RuntimeError(
        "No selected EOT embeddings found in memory. "
        "Run the previous EOT selection cell first."
    )

embedding_metric_eot_t = int(globals().get("embedding_metric_selected_eot_t", 0))
embedding_metric_selected_source = globals().get("embedding_metric_selected_eot_source")
if embedding_metric_selected_source != expected_selected_eot_source:
    raise RuntimeError(
        "Selected EOT embeddings were not computed from raw arrays. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_variant = globals().get("embedding_metric_selected_variant")
if embedding_metric_selected_variant != embedding_metric_variant:
    raise RuntimeError(
        "Selected EOT embeddings were computed for a different variant. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_eps_values = globals().get("embedding_metric_selected_eps_values")
if embedding_metric_selected_eps_values is None or not np.array_equal(
    np.asarray(embedding_metric_selected_eps_values, dtype=float),
    embedding_metric_eps_values,
):
    raise RuntimeError(
        "Selected EOT embeddings do not match the cached eps grid. "
        "Re-run the previous EOT selection cell."
    )

eot_metric_embeddings = np.asarray(eot_metric_embeddings, dtype=float)
expected_eot_shape = (
    len(embedding_metric_eps_values),
    ncov.joint_data.shape[0],
    2,
)
if eot_metric_embeddings.shape != expected_eot_shape:
    raise ValueError(
        f"Expected selected EOT embeddings with shape {expected_eot_shape}, "
        f"got {eot_metric_embeddings.shape}."
    )

metric_base_embeddings = globals().get("pca_embeddings")
if metric_base_embeddings is None:
    metric_base_embeddings = compute_pca_embedding_from_arrays(
        ncov.x1,
        ncov.x2,
        embd_dim=50,
        random_state=embedding_metric_random_state,
    )

metric_joint_tsne_embeddings = globals().get("tsne_embeddings")
if metric_joint_tsne_embeddings is None:
    metric_joint_tsne_embeddings = compute_tsne_embedding(
        metric_base_embeddings,
        random_state=embedding_metric_random_state,
    )

metric_specs = {
    "same_cell_type": {
        "label": "kNN Same-Cell-Type Fraction",
        "fn": lambda emb: compute_knn_same_cell_type_fraction(
            emb,
            ncov.joint_labels,
            k=embedding_metric_k,
        ),
        "cmap": "viridis",
    },
    "other_batch": {
        "label": "Shared-Type Batch Mixing Score",
        "fn": lambda emb: compute_knn_other_batch_fraction(
            emb,
            ncov.batch_ids,
            ncov.joint_labels,
            k=embedding_metric_k,
        ),
        "cmap": "viridis",
    },
}

eot_reference_label = f"EOT {embedding_metric_eot_t:g}"

heatmap_data_by_metric = {}
for metric_name, spec in metric_specs.items():
    matrix = np.empty((len(embedding_metric_eps_values), len(embedding_metric_percentages)), dtype=float)
    eot_column = np.empty(len(embedding_metric_eps_values), dtype=float)
    joint_tsne_value = float(spec["fn"](metric_joint_tsne_embeddings))
    joint_pca_value = float(spec["fn"](metric_base_embeddings))

    for eps_idx in range(len(embedding_metric_eps_values)):
        eot_column[eps_idx] = spec["fn"](eot_metric_embeddings[eps_idx])
        for pct_idx in range(len(embedding_metric_percentages)):
            matrix[eps_idx, pct_idx] = spec["fn"](
                ot_tsne_metric_embeddings[eps_idx, pct_idx]
            )

    reference_columns = [
        {"label": eot_reference_label, "values": eot_column},
        {
            "label": "joint t-SNE",
            "values": np.full(len(embedding_metric_eps_values), joint_tsne_value, dtype=float),
        },
        {
            "label": "joint PCA",
            "values": np.full(len(embedding_metric_eps_values), joint_pca_value, dtype=float),
        },
    ]

    heatmap_data_by_metric[metric_name] = {
        "metric_name": metric_name,
        "metric_label": spec["label"],
        "eps_times": embedding_metric_eps_values,
        "pct_values": 100.0 * embedding_metric_percentages,
        "eot_t": embedding_metric_eot_t,
        "dataset_names": ("nCoV",),
        "datasets": {
            "nCoV": {
                "top_pct_matrix": matrix,
                "reference_columns": reference_columns,
                "eot_column": eot_column,
            }
        },
        "y_labels": [f"{eps:g}" for eps in embedding_metric_eps_values],
        "top_pct_column_count": int(len(embedding_metric_percentages)),
    }

embedding_metric_scale_bounds = compute_shared_dataset_scale_bounds(heatmap_data_by_metric)
embedding_metric_heatmap_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_knn_metric_heatmaps_{embedding_metric_variant}.npz"
)

for metric_name, heatmap_data in heatmap_data_by_metric.items():
    figure_path = (
        embedding_metric_figure_dir
        / f"{metric_name}_joint_ot_tsne_eps_pct_heatmap_{embedding_metric_variant}.png"
    )
    fig, axes = plot_metric_heatmap_grid(
        heatmap_data,
        title=(
            f"{heatmap_data['metric_label']} on joint nCoV OT-tSNE embeddings "
            f"({embedding_metric_variant}, k={embedding_metric_k})"
        ),
        cmap=metric_specs[metric_name]["cmap"],
        save_path=figure_path,
        scale_bounds_by_dataset=embedding_metric_scale_bounds,
        annotate_cells=True,
    )
    axes["nCoV"]["main"].set_ylabel("eps_times")
    plt.show()
    print(f"Saved {metric_name} heatmap figure to {figure_path}")

np.savez_compressed(
    embedding_metric_heatmap_cache_path,
    eps_values=embedding_metric_eps_values,
    percentages=embedding_metric_percentages,
    pct_values_percent=100.0 * embedding_metric_percentages,
    metric_k=np.asarray(embedding_metric_k, dtype=int),
    eot_t=np.asarray(embedding_metric_eot_t, dtype=int),
    eot_source=np.asarray(expected_selected_eot_source),
    binarize=np.asarray(embedding_metric_binarize),
    same_cell_type_matrix=heatmap_data_by_metric["same_cell_type"]["datasets"]["nCoV"]["top_pct_matrix"],
    same_cell_type_eot=heatmap_data_by_metric["same_cell_type"]["datasets"]["nCoV"]["eot_column"],
    same_cell_type_joint_tsne=heatmap_data_by_metric["same_cell_type"]["datasets"]["nCoV"]["reference_columns"][1]["values"],
    same_cell_type_joint_pca=heatmap_data_by_metric["same_cell_type"]["datasets"]["nCoV"]["reference_columns"][2]["values"],
    other_batch_matrix=heatmap_data_by_metric["other_batch"]["datasets"]["nCoV"]["top_pct_matrix"],
    other_batch_eot=heatmap_data_by_metric["other_batch"]["datasets"]["nCoV"]["eot_column"],
    other_batch_joint_tsne=heatmap_data_by_metric["other_batch"]["datasets"]["nCoV"]["reference_columns"][1]["values"],
    other_batch_joint_pca=heatmap_data_by_metric["other_batch"]["datasets"]["nCoV"]["reference_columns"][2]["values"],
)
print(f"Saved embedding-metric heatmap matrices to {embedding_metric_heatmap_cache_path}")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

embedding_metric_cache_dir_candidates = (Path("data/nCoV"), Path("../data/nCoV"))
embedding_metric_cache_dir = next(
    (path for path in embedding_metric_cache_dir_candidates if path.exists()),
    embedding_metric_cache_dir_candidates[0],
)
embedding_metric_figure_dir = embedding_metric_cache_dir / "figures"
embedding_metric_figure_dir.mkdir(parents=True, exist_ok=True)

embedding_metric_variant = globals().get("embedding_metric_variant", "binarized")
embedding_metric_cache_path = (
    embedding_metric_cache_dir
    / f"joint_ot_tsne_sweep_embeddings_{embedding_metric_variant}.npz"
)
if not embedding_metric_cache_path.exists():
    raise FileNotFoundError(
        f"No saved OT-tSNE embedding cache found at {embedding_metric_cache_path}. "
        "Run the embedding cache cell first."
    )

with np.load(embedding_metric_cache_path) as cache:
    embedding_metric_eps_values = np.asarray(cache["eps_values"], dtype=float)

expected_selected_eot_source = "raw_arrays"
eot_metric_embeddings = globals().get("embedding_metric_selected_eot_embeddings")
if eot_metric_embeddings is None:
    raise RuntimeError(
        "No selected EOT embeddings found in memory. "
        "Run the previous EOT selection cell first."
    )

embedding_metric_eot_t = int(globals().get("embedding_metric_selected_eot_t", 0))
embedding_metric_selected_source = globals().get("embedding_metric_selected_eot_source")
if embedding_metric_selected_source != expected_selected_eot_source:
    raise RuntimeError(
        "Selected EOT embeddings were not computed from raw arrays. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_variant = globals().get("embedding_metric_selected_variant")
if embedding_metric_selected_variant != embedding_metric_variant:
    raise RuntimeError(
        "Selected EOT embeddings were computed for a different variant. "
        "Re-run the previous EOT selection cell."
    )

embedding_metric_selected_eps_values = globals().get("embedding_metric_selected_eps_values")
if embedding_metric_selected_eps_values is None or not np.array_equal(
    np.asarray(embedding_metric_selected_eps_values, dtype=float),
    embedding_metric_eps_values,
):
    raise RuntimeError(
        "Selected EOT embeddings do not match the cached eps grid. "
        "Re-run the previous EOT selection cell."
    )

eot_metric_embeddings = np.asarray(eot_metric_embeddings, dtype=float)
expected_eot_shape = (
    len(embedding_metric_eps_values),
    ncov.joint_data.shape[0],
    2,
)
if eot_metric_embeddings.shape != expected_eot_shape:
    raise ValueError(
        f"Expected selected EOT embeddings with shape {expected_eot_shape}, "
        f"got {eot_metric_embeddings.shape}."
    )

cell_type_palette = np.array(
    [plt.get_cmap("tab20", len(ncov.cell_type_names))(i) for i in range(len(ncov.cell_type_names))]
)
batch_palette = np.array(["tab:red", "tab:blue"], dtype=object)
plot_order = np.random.default_rng(0).permutation(ncov.joint_data.shape[0])


def standardize_embedding(embedding, eps=np.finfo(float).eps):
    mean = embedding.mean(axis=0, keepdims=True)
    std = embedding.std(axis=0, keepdims=True)
    std = np.where(std < eps, 1.0, std)
    return (embedding - mean) / std


fig, axes = plt.subplots(
    len(embedding_metric_eps_values),
    2,
    figsize=(12, 3.2 * len(embedding_metric_eps_values)),
    constrained_layout=True,
    squeeze=False,
)

for row_idx, eps_times in enumerate(embedding_metric_eps_values):
    embedding = standardize_embedding(eot_metric_embeddings[row_idx])
    x_min, x_max = np.quantile(embedding[:, 0], [0.01, 0.99])
    y_min, y_max = np.quantile(embedding[:, 1], [0.01, 0.99])
    x_pad = 0.08 * max(x_max - x_min, 1.0)
    y_pad = 0.08 * max(y_max - y_min, 1.0)

    axes[row_idx, 0].scatter(
        embedding[plot_order, 0],
        embedding[plot_order, 1],
        s=1.5,
        c=cell_type_palette[ncov.joint_labels][plot_order],
        alpha=0.7,
    )
    axes[row_idx, 1].scatter(
        embedding[plot_order, 0],
        embedding[plot_order, 1],
        s=1.5,
        c=batch_palette[ncov.batch_ids][plot_order],
        alpha=0.7,
    )

    if row_idx == 0:
        axes[row_idx, 0].set_title("Cell type")
        axes[row_idx, 1].set_title("Batch")

    axes[row_idx, 0].set_ylabel(f"eps_times={eps_times:g}")

    for ax in axes[row_idx]:
        ax.set_xlim(x_min - x_pad, x_max + x_pad)
        ax.set_ylim(y_min - y_pad, y_max + y_pad)
        ax.set_aspect("equal")
        ax.axis("off")

fig.suptitle(
    f"Paper-style EOT t={embedding_metric_eot_t} reference embeddings from raw arrays used in the embedding heatmaps"
)

eot_reference_figure_path = (
    embedding_metric_figure_dir
    / f"eot_t{embedding_metric_eot_t}_raw_arrays_reference_embeddings_{embedding_metric_variant}.png"
)
fig.savefig(eot_reference_figure_path, bbox_inches="tight")
plt.show()
print(f"Saved EOT reference embedding figure to {eot_reference_figure_path}")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ne_aligned_utils.methods import (
    compute_eot_embedding_from_plan,
    compute_ot_plan,
    compute_pca_embedding_from_arrays,
    compute_seurat_embedding_from_arrays,
    compute_tsne_embedding,
)
from ne_aligned_utils.metrics import compute_neighbor_purity, compute_silhouette_index

comparison_q_values = tuple(range(2, 21))
comparison_metric_k = 50
comparison_eot_t = int(globals().get("embedding_metric_selected_eot_t", 0))
comparison_eot_eps_times = float(globals().get("embedding_scatter_eot_eps_times", 1.0))
comparison_title = "COVID PBMCs scRNA-seq"

data_obj = ncov
labels = np.asarray(data_obj.joint_labels, dtype=int)

pca_full = compute_pca_embedding_from_arrays(
    data_obj.x1,
    data_obj.x2,
    embd_dim=max(comparison_q_values),
    random_state=0,
)
tsne_embedding = compute_tsne_embedding(pca_full, random_state=0)
ot_plan = compute_ot_plan(data_obj.x1, data_obj.x2, eps_times=comparison_eot_eps_times)


def compute_metric_pair(embedding):
    embedding = np.asarray(embedding, dtype=float)
    purity = compute_neighbor_purity(embedding, labels, k=comparison_metric_k)
    silhouette = compute_silhouette_index(embedding, labels)
    return float(purity), float(silhouette)


def summarize_over_q(label, embedding_builder, aggregation_note):
    rows = []
    for q in comparison_q_values:
        embedding = embedding_builder(int(q))
        purity, silhouette = compute_metric_pair(embedding)
        rows.append((int(q), float(purity), float(silhouette)))
    values = np.asarray([(row[1], row[2]) for row in rows], dtype=float)
    return {
        "label": label,
        "neighbor_purity": float(np.median(values[:, 0])),
        "silhouette_index": float(np.median(values[:, 1])),
        "aggregation": aggregation_note,
        "per_q_rows": rows,
    }

comparison_points = [
    summarize_over_q(
        "j-pca",
        lambda q: pca_full[:, :q],
        f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}",
    ),
    summarize_over_q(
        "seurat",
        lambda q: compute_seurat_embedding_from_arrays(data_obj.x1, data_obj.x2, embd_dim=q),
        f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}",
    ),
    summarize_over_q(
        "EOT",
        lambda q: compute_eot_embedding_from_plan(
            ot_plan,
            embd_dim=q,
            t=comparison_eot_t,
        ),
        (
            f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}, "
            f"t={comparison_eot_t}, eps_times={comparison_eot_eps_times:g}"
        ),
    ),
]

joint_tsne_purity, joint_tsne_silhouette = compute_metric_pair(tsne_embedding)
comparison_points.append(
    {
        "label": "j-tsne",
        "neighbor_purity": joint_tsne_purity,
        "silhouette_index": joint_tsne_silhouette,
        "aggregation": "single 2D embedding",
    }
)

comparison_df = pd.DataFrame(
    [{key: value for key, value in point.items() if key != "per_q_rows"} for point in comparison_points]
)
display(comparison_df.round(4))

fig, ax = plt.subplots(figsize=(7.5, 5.8), constrained_layout=True)
for point in comparison_points:
    ax.scatter(point["neighbor_purity"], point["silhouette_index"], color="black", s=30)
    ax.annotate(
        point["label"],
        (point["neighbor_purity"], point["silhouette_index"]),
        xytext=(6, 4),
        textcoords="offset points",
    )
ax.set_title(comparison_title)
ax.set_xlabel("median neighbor purity")
ax.set_ylabel("median silhouette index")
ax.grid(alpha=0.25)
plt.show()


In [ ]:
from ne_aligned_utils.metrics import compute_kbet

comparison_kbet_metric_name = "kBET"
comparison_kbet_selector = "kbet_plus_silhouette"
batch_ids = np.asarray(ncov.batch_ids, dtype=int)


def score_kbet_silhouette_pair(embedding):
    embedding = np.asarray(embedding, dtype=float)
    kbet = compute_kbet(embedding, batch_ids, labels)
    silhouette = compute_silhouette_index(embedding, labels)
    return float(kbet), float(silhouette)


def summarize_metric_over_q(label, embedding_builder, aggregation_note):
    kbet_values = []
    silhouette_values = []
    per_q_rows = []
    for q in comparison_q_values:
        embedding = embedding_builder(q)
        kbet, silhouette = score_kbet_silhouette_pair(embedding)
        kbet_values.append(kbet)
        silhouette_values.append(silhouette)
        per_q_rows.append((int(q), float(kbet), float(silhouette)))

    return {
        "label": label,
        "kbet": float(np.nanmedian(kbet_values)),
        "silhouette_index": float(np.nanmedian(silhouette_values)),
        "aggregation": aggregation_note,
        "per_q_rows": per_q_rows,
    }


kbet_style_points = []
kbet_style_points.append(
    summarize_metric_over_q(
        "j-pca",
        lambda q: pca_embeddings[:, : min(int(q), pca_embeddings.shape[1])],
        f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}",
    )
)
kbet_style_points.append(
    summarize_metric_over_q(
        "seurat",
        lambda q: np.concatenate(
            [
                seurat_u[:, : min(int(q), seurat_u.shape[1], seurat_vt.shape[0])],
                seurat_vt.T[:, : min(int(q), seurat_u.shape[1], seurat_vt.shape[0])],
            ],
            axis=0,
        ),
        f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}",
    )
)
kbet_style_points.append(
    summarize_metric_over_q(
        "EOT",
        build_eot_embedding,
        (
            f"median over q={comparison_q_values[0]}..{comparison_q_values[-1]}, "
            f"t={comparison_eot_t}, eps_times={comparison_eot_eps_times:g}, raw-array OT plan"
        ),
    )
)

kbet_comparison_points = list(kbet_style_points)

joint_tsne_kbet, joint_tsne_silhouette = score_kbet_silhouette_pair(tsne_embeddings)
kbet_comparison_points.append(
    {
        "label": "j-tsne",
        "kbet": joint_tsne_kbet,
        "silhouette_index": joint_tsne_silhouette,
        "aggregation": "single 2D embedding",
    }
)

best_kbet_ot_tsne = None
for eps_idx, eps_times in enumerate(ot_tsne_eps_values):
    for pct_idx, percentage in enumerate(ot_tsne_percentages):
        embedding = ot_tsne_metric_embeddings[eps_idx, pct_idx]
        kbet, silhouette = score_kbet_silhouette_pair(embedding)
        composite_score = float(kbet + silhouette)
        candidate = {
            "label": "best OT-tSNE",
            "kbet": kbet,
            "silhouette_index": silhouette,
            "aggregation": (
                "single 2D embedding selected from cached sweep by maximizing "
                "kBET + silhouette_index"
            ),
            "eps_times": float(eps_times),
            "percentage": float(percentage),
            "composite_score": composite_score,
        }
        if best_kbet_ot_tsne is None or composite_score > best_kbet_ot_tsne["composite_score"]:
            best_kbet_ot_tsne = candidate

kbet_comparison_points.append(best_kbet_ot_tsne)

fig, ax = plt.subplots(figsize=(8.2, 6.6), constrained_layout=True)
fig.patch.set_facecolor("white")
ax.set_facecolor("#EBEBEB")
ax.grid(color="white", linewidth=1.0)
ax.set_axisbelow(True)

for point in kbet_comparison_points:
    x = point["kbet"]
    y = point["silhouette_index"]
    ax.scatter(x, y, color="black", s=28, zorder=3)
    ax.annotate(
        point["label"],
        (x, y),
        xytext=label_offsets.get(point["label"], (6, 6)),
        textcoords="offset points",
        ha=label_alignments.get(point["label"], "left"),
        va="center",
        fontsize=11,
        fontweight="bold",
        color="black",
    )

x_values = np.array([point["kbet"] for point in kbet_comparison_points], dtype=float)
y_values = np.array([point["silhouette_index"] for point in kbet_comparison_points], dtype=float)
x_pad = max(0.01, 0.08 * float(np.nanmax(x_values) - np.nanmin(x_values) or 1.0))
y_pad = max(0.005, 0.10 * float(np.nanmax(y_values) - np.nanmin(y_values) or 1.0))
ax.set_xlim(max(0.0, float(np.nanmin(x_values) - x_pad)), min(1.0, float(np.nanmax(x_values) + x_pad)))
ax.set_ylim(float(np.nanmin(y_values) - y_pad), float(np.nanmax(y_values) + y_pad))

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title(comparison_title, fontsize=22, fontweight="bold", pad=10)
ax.set_xlabel("median kBET", fontsize=16, fontweight="bold")
ax.set_ylabel("median silhouette index", fontsize=16, fontweight="bold")
ax.tick_params(labelsize=11)

comparison_kbet_figure_path = (
    comparison_figure_dir
    / f"paper_style_kbet_silhouette_scatter_{comparison_variant}.png"
)
fig.savefig(comparison_kbet_figure_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved paper-style kBET/silhouette scatter to {comparison_kbet_figure_path}")
print(
    f"Best OT-tSNE sweep point: eps_times={best_kbet_ot_tsne['eps_times']:g}, "
    f"top_pct={100.0 * best_kbet_ot_tsne['percentage']:g}%"
)
print(
    "Note: j-pca, seurat, and EOT use the paper's median-over-q summary; "
    "j-tsne and best OT-tSNE are single 2D points."
)
print()
for point in kbet_comparison_points:
    extra = ""
    if point["label"] == "best OT-tSNE":
        extra = (
            f", eps_times={point['eps_times']:g}, "
            f"top_pct={100.0 * point['percentage']:g}%"
        )
    print(
        f"{point['label']:>13s}: kBET={point['kbet']:.6f}, "
        f"silhouette={point['silhouette_index']:.6f} "
        f"[{point['aggregation']}{extra}]"
    )
